In [1]:
def score_to_label(score):
    if score < 4:
        return "Poor"
    elif score < 7:
        return "Moderate"
    elif score < 9:
        return "Good"
    else:
        return "Excellent"
    
#backend için

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ======================
# LOAD DATA
# ======================
df = pd.read_csv("../../data/hourly_observations.csv")

# ======================
# FEATURE SET (ONLY NUMERIC)
# ======================
#outdoor_suitability_score için
features = [
    "temperature_c",
    "feels_like_c",
    "dew_point_c",
    "humidity_pct",
    "pressure_hpa",
    "wind_speed_kmh",
    "wind_direction_deg",
    "wind_gust_kmh",
    "cloud_cover_pct",
    "visibility_km",
    "uv_index",
    "hour_of_day",
    "month",
]

X = df[features]
y = df["outdoor_suitability_score"]

# ======================
# TRAIN / TEST SPLIT
# IMPORTANT: split first, then balance only training data
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ======================
# BALANCE ONLY TRAINING SET
# ======================
train_df = X_train.copy()
train_df["outdoor_suitability_score"] = y_train.values

target_per_class = 2000
balanced_parts = []

for score, group in train_df.groupby("outdoor_suitability_score"):
    n = len(group)

    if n >= target_per_class:
        # downsample large classes
        sampled = group.sample(n=target_per_class, replace=False, random_state=42)
    else:
        # controlled upsample for small classes
        sampled = group.sample(
            n=min(target_per_class, n * 5),
            replace=True,
            random_state=42
        )

    balanced_parts.append(sampled)

train_balanced = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)

X_train_balanced = train_balanced[features]
y_train_balanced = train_balanced["outdoor_suitability_score"]

print("\n=== Original Train Class Distribution ===")
print(y_train.value_counts().sort_index())

print("\n=== Balanced Train Class Distribution ===")
print(y_train_balanced.value_counts().sort_index())

print("\n=== Test Class Distribution (untouched) ===")
print(y_test.value_counts().sort_index())

# ======================
# MODEL
# ======================
model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# ======================
# TRAIN
# ======================
model.fit(X_train_balanced, y_train_balanced)

# ======================
# TEST PREDICTION
# ======================
y_pred = model.predict(X_test)

# Optional: round predictions to nearest score bucket
y_pred_rounded = np.clip(np.rint(y_pred), 0, 10)

# ======================
# METRICS
# ======================
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

mae_rounded = mean_absolute_error(y_test, y_pred_rounded)
rmse_rounded = np.sqrt(mean_squared_error(y_test, y_pred_rounded))
r2_rounded = r2_score(y_test, y_pred_rounded)

print("\n=== Test Metrics (Raw Prediction) ===")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

print("\n=== Test Metrics (Rounded Prediction) ===")
print("MAE:", mae_rounded)
print("RMSE:", rmse_rounded)
print("R2:", r2_rounded)

# ======================
# CROSS VALIDATION
# NOTE: CV is done on balanced training data only
# ======================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mae_scores = -cross_val_score(
    model,
    X_train_balanced,
    y_train_balanced,
    cv=kf,
    scoring="neg_mean_absolute_error"
)

cv_rmse_scores = np.sqrt(
    -cross_val_score(
        model,
        X_train_balanced,
        y_train_balanced,
        cv=kf,
        scoring="neg_mean_squared_error"
    )
)

cv_r2_scores = cross_val_score(
    model,
    X_train_balanced,
    y_train_balanced,
    cv=kf,
    scoring="r2"
)

print("\n=== Cross Validation on Balanced Train Set ===")
print("CV MAE Scores:", cv_mae_scores)
print("Mean CV MAE:", cv_mae_scores.mean())

print("\nCV RMSE Scores:", cv_rmse_scores)
print("Mean CV RMSE:", cv_rmse_scores.mean())

print("\nCV R2 Scores:", cv_r2_scores)
print("Mean CV R2:", cv_r2_scores.mean())

# ======================
# FEATURE IMPORTANCE
# ======================
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\n=== Feature Importance ===")
print(feature_importance)

# ======================
# SAMPLE PREDICTION
# ======================
sample = X_test.iloc[[2]]
true_score = y_test.iloc[2]
pred_score = model.predict(sample)[0]
pred_score_rounded = int(np.clip(np.rint(pred_score), 0, 10))

print("\n=== Sample Prediction ===")
print("True score:", true_score)
print("Predicted score (raw):", round(pred_score, 2))
print("Predicted score (rounded):", pred_score_rounded)


=== Original Train Class Distribution ===
outdoor_suitability_score
0      2486
2      1116
3         5
4       861
5        36
6     11390
7         5
8      2476
9        59
10    16318
Name: count, dtype: int64

=== Balanced Train Class Distribution ===
outdoor_suitability_score
0     2000
2     2000
3       25
4     2000
5      180
6     2000
7       25
8     2000
9      295
10    2000
Name: count, dtype: int64

=== Test Class Distribution (untouched) ===
outdoor_suitability_score
0      636
2      257
4      216
5        6
6     2768
7        2
8      623
9       18
10    4162
Name: count, dtype: int64

=== Test Metrics (Raw Prediction) ===
MAE: 0.21971538662910461
RMSE: 0.7013467979601185
R2: 0.9456673860549927

=== Test Metrics (Rounded Prediction) ===
MAE: 0.15515653789043427
RMSE: 0.7142330959217068
R2: 0.9436525106430054

=== Cross Validation on Balanced Train Set ===
CV MAE Scores: [0.16473462 0.18018416 0.17394207 0.18807489 0.18412665]
Mean CV MAE: 0.17821247577667237

CV

In [3]:
import joblib

joblib.dump(model, "../../models/suitability_model.pkl")

['../../models/suitability_model.pkl']

In [4]:
import pandas as pd

df = pd.read_csv("../../data/hourly_observations.csv")
print(df["outdoor_suitability_score"].value_counts())

outdoor_suitability_score
10    20480
6     14158
0      3122
8      3099
2      1373
4      1077
9        77
5        42
7         7
3         5
Name: count, dtype: int64


In [5]:
sample ={
  "temperature_c": 18.0,
  "feels_like_c": 17.0,
  "dew_point_c": 12.0,
  "humidity_pct": 70.0,
  "pressure_hpa": 1012.0,
  "wind_speed_kmh": 15.0,
  "wind_direction_deg": 180.0,
  "wind_gust_kmh": 22.0,
  "cloud_cover_pct": 90.0,
  "visibility_km": 8.0,
  "uv_index": 2.0,
  "hour_of_day": 14,
  "month": 4,
}
sample_df = pd.DataFrame([sample])
prediction = model.predict(sample_df)
print(prediction)
pred = int(prediction[0])

[5.960319]
